<a href="https://colab.research.google.com/github/VizDataWorks/eCommerce_data_analysis/blob/main/ecommerce-challenge/notebooks/01_order_volume_by_hour.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · Order volume by hour

**Question:** What is order volume by hour (based on company time zone)? Does this order volume support the need to maintain 24/7 operational staffing for customer fulfillment?

**Short answer:** **No.** Only 8.6% of orders arrive between midnight and 6 AM Central, about 3–6 an hour, against about 21 an hour at the 4 PM peak. Full fulfillment staffing from about 6 AM to midnight, with a light overnight presence, fits the volume.

**Approach**
- **Company time zone = Central (`America/Chicago`)**, since the company is based in Missouri.
- **`order_datetime` is the buyer's local time.** Order hours peak at the same local hour in every buyer time zone, so the cleaning notebook converts each order to Central (`order_datetime_company`), using the buyer's time zone from the customer table. Daylight saving is handled automatically.
- **One order = one count.** The sales table has one row per order + product, so lines are reduced to one row per `order_id`.
- **Only orders with a real time are used.** Date-only values (e.g. `6/26/24`) and the `1/1/70` placeholder have no hour, so they are left out.
- **All statuses are counted.** The question is about orders arriving, not revenue.

## 1. Setup

In [ ]:
import os, sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Get the project data from GitHub: no Google Drive or sign-in needed
REPO = "https://github.com/VizDataWorks/CapitalOne_DataAnalysis.git"
!rm -rf repo
!git clone -q --depth 1 {REPO} repo

CLEAN = os.path.join("repo", "ecommerce-challenge", "data", "clean")

OVERNIGHT = range(0, 6)          # midnight to 6 AM

pd.set_option("display.max_columns", 30, "display.width", 160)

## 2. Load data

`sales_clean.csv` already holds each line's order time in company time (`order_datetime_company`), converted from the buyer's local time in the cleaning notebook. Lines with no time of day are blank there.

In [3]:
sales = pd.read_csv(os.path.join(CLEAN, "sales.csv.gz"),
                    usecols=["order_id", "order_datetime_company"], parse_dates=["order_datetime_company"])

# Every line of an order has the same time, so keep one row per order
orders = sales.groupby("order_id").order_datetime_company.min()
print(f"{orders.notna().sum():,} of {len(orders):,} orders have a time of day; "
      f"{orders.isna().mean():.1%} are left out (no time of day, or buyer has no time zone)")
orders = orders.dropna()

227,958 of 244,280 orders have a time of day; 6.7% are left out (no time of day, or buyer has no time zone)


## 3. Order volume by hour (Central time)

Each hour shows total orders, its share of the day, the average per day, a busy day (95th percentile) and the number of days with no orders at all. Hours with no orders count as 0, so the averages aren't inflated.

In [ ]:
day = orders.dt.normalize().rename("day")
hour = orders.dt.hour.rename("hour")
all_days = pd.date_range(day.min(), day.max())

# Orders per day x hour. Empty slots count as 0 so the averages aren't inflated
daily = pd.crosstab(day, hour).reindex(index=all_days, columns=range(24), fill_value=0)

hourly = pd.DataFrame({
    "orders": daily.sum(),
    "share": daily.sum() / daily.sum().sum(),
    "avg_per_day": daily.mean(),
    "busy_day_p95": daily.quantile(0.95),
    "days_with_0": (daily == 0).sum(),
})
print(f"{len(orders):,} orders over {len(all_days)} days, about {len(orders) / len(all_days):.0f} a day")

hourly.style.format({"orders": "{:,}", "share": "{:.1%}", "avg_per_day": "{:.1f}", "busy_day_p95": "{:.0f}"}) \
    .background_gradient(subset=["avg_per_day"], cmap="Blues")

227,958 orders over 731 days (2024-01-01 to 2025-12-31), about 312 orders a day


,orders,share,avg_per_day,busy_day_p95,days_with_0
hour_ct,,,,,
0,"4,177",1.8%,5.7,11,9
1,"2,986",1.3%,4.1,9,35
2,"2,156",0.9%,2.9,8,76
3,"2,518",1.1%,3.4,8,33
4,"3,304",1.4%,4.5,10,11
5,"4,458",2.0%,6.1,12,5
6,"6,398",2.8%,8.8,17,1
7,"8,335",3.7%,11.4,22,2
8,"9,868",4.3%,13.5,25,1


In [4]:
# Chart: average orders per hour, with a busy day (p95) marked on each bar

ax = hourly.avg_per_day.plot.bar(figsize=(11, 4), width=0.8, color="#2a78d6", label="Average day")
ax.scatter(hourly.index, hourly.busy_day_p95, marker="_", s=150, color="black", zorder=3, label="Busy day (p95)")
ax.axvspan(-0.5, 5.5, color="grey", alpha=0.15, zorder=0, label="Overnight (12–6 AM)")
ax.set(title="Orders per hour, Central time (2024–2025)", xlabel="Hour of day (CT)", ylabel="Orders per hour")
ax.tick_params(axis="x", rotation=0)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper left")
plt.show()

NameError: name 'hourly' is not defined

## 4. Overnight vs daytime

In [ ]:
night = hourly.loc[list(OVERNIGHT)]
day_hours = hourly.drop(index=list(OVERNIGHT))
peak, quiet = hourly.avg_per_day.idxmax(), hourly.avg_per_day.idxmin()

summary = pd.Series({
    "Overnight (12–6 AM) share of orders": f"{night.orders.sum() / hourly.orders.sum():.1%}",
    "Daytime (6 AM–12 AM) share of orders": f"{day_hours.orders.sum() / hourly.orders.sum():.1%}",
    "Overnight orders per hour (average day)": f"{night.avg_per_day.min():.1f} – {night.avg_per_day.max():.1f}",
    "Overnight orders per hour (busy day, p95)": f"{night.busy_day_p95.min():.0f} – {night.busy_day_p95.max():.0f}",
    "Overnight orders per night (average)": f"{night.avg_per_day.sum():.0f}",
    "Peak hour (average day)": f"{peak}:00 CT, {hourly.avg_per_day[peak]:.1f} orders",
    "Quietest hour (average day)": f"{quiet}:00 CT, {hourly.avg_per_day[quiet]:.1f} orders",
    "Peak vs quietest hour": f"{hourly.avg_per_day[peak] / hourly.avg_per_day[quiet]:.1f}x",
    f"Days with no orders at {quiet}:00 CT": f"{hourly.days_with_0[quiet]}",
}, name="value")
summary.to_frame()

,value
Overnight (12–6 AM) share of orders,8.6%
Daytime (6 AM–12 AM) share of orders,91.4%
Overnight orders per hour (average day),2.9 – 6.1
"Overnight orders per hour (busy day, p95)",8 – 12
Overnight orders per night (average),27
Peak hour (average day),"16:00 CT, 20.7 orders"
Quietest hour (average day),"2:00 CT, 2.9 orders"
Peak vs quietest hour,7.0x
Days with no orders at 2:00 CT,76


## 5. Does peak demand change the picture?

If 24/7 staffing were needed anywhere, it would be on the busiest days and on weekends. These checks look at overnight volume on those days.

In [5]:
by_day = pd.DataFrame({
    "orders": daily.sum(axis=1),
    "overnight_per_hour": daily[OVERNIGHT].mean(axis=1),
    "daytime_per_hour": daily.drop(columns=OVERNIGHT).mean(axis=1),
})

# Five busiest days
top5 = by_day.nlargest(5, "orders")
top5.insert(0, "weekday", top5.index.day_name())
top5["vs_average_day"] = top5.orders / by_day.orders.mean()
display(top5.round(1))

# Weekdays vs weekends (averages per day)
weekend = pd.Series(by_day.index.dayofweek >= 5, index=by_day.index).map({False: "Weekday", True: "Weekend"})
display(by_day.groupby(weekend).mean().round(1))

NameError: name 'daily' is not defined

## 6. Conclusion

**Order volume does not support full 24/7 fulfillment staffing.**

- **Volume is lopsided.** 6 AM to midnight CT holds 91.4% of orders. The peak hour (4 PM, 20.7 orders on an average day) is 7× the quietest (2 AM, 2.9 orders). On 76 days, no order at all arrived between 2 and 3 AM.
- **Overnight stays thin even on busy days.** On a busy day (95th percentile), midnight to 6 AM sees only 8–12 orders an hour. Black Friday brings about 4× a normal day's orders, yet its overnight hours averaged 14–16 orders an hour, against 62–65 during the day.
- **Weekends are busier, but weekend nights aren't.** Weekends have about 1.8× a weekday's orders (459 vs 253 a day), and weekend overnight still averages about 7 orders an hour.

**A staffing model that fits the volume**
- **Full staffing about 6 AM to midnight CT**, with day and evening shifts overlapping roughly 2–8 PM to cover the peak.
- **Overnight: an on-call person or a small skeleton crew.** The roughly 27 orders placed on an average night can be cleared in the first morning wave.
- **Surge staffing for Black Friday through Cyber Monday and on weekends**, added mainly to the afternoon and evening shifts.

**Limits**
- This measures when orders are **placed**, not fulfillment work. The data has no pick, pack or ship times and no delivery promise. If the company promises same-day or next-morning shipping, some overnight fulfillment could still be needed.
- 6.7% of orders are left out because they have no time of day or no buyer time zone.